# OpenSeesPy

![gallery_thumbnail](/_static/images/gallery/opensees-3a8cf154.png)


Xara provides a compatibility module that can be used to run OpenSeesPy scripts. 

Change the import:
```python
from openseespy.opensees import *
```
to:
```python
from opensees.openseespy import *
```

For example, the following script applies this change to an OpenSeesPy truss model:

In [1]:
from opensees.openseespy import *

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------
# Start of model generation
# -----------------------------

# remove existing model
wipe()

# set modelbuilder
model('basic', '-ndm', 2, '-ndf', 2)

# create nodes
node(1, 0.0, 0.0)
node(2, 144.0,  0.0)
node(3, 168.0,  0.0)
node(4,  72.0, 96.0)

# set boundary condition
fix(1, 1, 1)
fix(2, 1, 1)
fix(3, 1, 1)

# define materials
uniaxialMaterial("Elastic", 1, 3000.0)

# define elements
element("Truss",1,1,4,10.0,1)
element("Truss",2,2,4,5.0,1)
element("Truss",3,3,4,5.0,1)

# create TimeSeries
timeSeries("Linear", 1)

# create a plain load pattern
pattern("Plain", 1, 1)

# Create the nodal load - command: load nodeID xForce yForce
load(4, 100.0, -50.0)

# ------------------------------
# Start of analysis generation
# ------------------------------

# create SOE
system("BandSPD")

# create DOF number
numberer("RCM")

# create constraint handler
constraints("Plain")

# create integrator
integrator("LoadControl", 1.0)

# create algorithm
algorithm("Linear")

# create analysis object
analysis("Static")

# perform the analysis
analyze(1)

ux = nodeDisp(4,1)
uy = nodeDisp(4,2)
if abs(ux-0.53009277713228375450)<1e-12 and abs(uy+0.17789363846931768864)<1e-12:
    print("Passed!")
else:
    print("Failed!")

Passed!


However, a better way to run this simulation leverages xara-specific features:

In [3]:

import xara, veux

#
# Define Model
#

# Create a Model (with two-dimensions and 2 DOF/node)
model = xara.Model(ndm=2, ndf=2)

# Create nodes
model.node(1, (  0.0,  0.0))
model.node(2, (144.0,  0.0))
model.node(3, (168.0,  0.0))
model.node(4, ( 72.0, 96.0))

# Set the boundary conditions
model.fix(1, (1, 1))
model.fix(2, (1, 1))
model.fix(3, (1, 1))

# Define materials for truss elements
# -----------------------------------
model.uniaxialMaterial("Elastic", 1, 3000.0)

# Define elements
# ---------------
# Create truss elements - command: element truss tag, (node1, node2), area, material
model.element("Truss", 1, (1, 4), 10.0, 1)
model.element("Truss", 2, (2, 4),  5.0, 1)
model.element("Truss", 3, (3, 4),  5.0, 1)

# Define loads
# ------------
# Define the load at node 4 with components 100 and -50 in x and y:
load = {4: [100, -50.0]}

# Assign the load to a "Plain" load pattern and scale its load factor linearly in time.
model.pattern("Plain", 1, "Linear", load=load)


#
# Define Analysis
#

# create the solution algorithm, a Linear algorithm is created
model.algorithm("Linear")

# create the integration scheme, the LoadControl scheme using steps of 1.0
model.integrator("LoadControl", 1.0)
model.analysis("Static")


#
# Finally perform the analysis
#

model.analyze(1)

#
# Print results
#

# print the current state at node 4 and at all elements

ux = model.nodeDisp(4,1)
uy = model.nodeDisp(4,2)
if abs(ux-0.53009277713228375450)<1e-12 and abs(uy+0.17789363846931768864)<1e-12:
    print("Passed!")
else:
    print("Failed!")


Passed!
